### Imports and R Environment Setup


In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
import warnings

# Filter out the specific rpy2 environment variable warnings
warnings.filterwarnings("ignore", category=UserWarning, message='.*Environment variable ".*" redefined by R.*')

In [33]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# Core project imports
from graphical_sampling.sampling import KMeansSampler
from graphical_sampling.population import Population
from package_sampling.utils import inclusion_probabilities

# Your Python-based index modules from the index folder
from graphical_sampling.index import Density
from graphical_sampling.index import Moran 
from graphical_sampling.index import Voronoi 
from graphical_sampling.index import LocalBalance 

In [34]:
def evaluate_sampler(sampler: KMeansSampler, y_values: np.ndarray):
    """
    Calculates the exact theoretical metrics for the joint design.
    Returns expected values, standard deviations, and HT variance.
    """
    # 1. Access the joint design data
    all_samples = sampler.all_samples       # Every possible sample
    all_probs = sampler.all_samples_probs   # Exact probability of each sample
    n = all_samples.shape[1]
    N = len(y_values)
    # 2. Get population data for HT calculation
    pik = sampler.population.probs          # Inclusion probabilities
    true_total = np.sum(y_values)           # The real population total

    # 3. Initialize Python-side scorers
    moran_scorer = Moran(population=sampler.population)
    voronoi_scorer = Voronoi(population=sampler.population)
    lb_scorer = LocalBalance(population=sampler.population)
    density_scorer = Density(population=sampler.population, k=n)
    
    results = []
    
    # 4. Loop through ALL possible samples
    sum_prob = 0
    for i in tqdm(range(len(all_samples)), desc="Theoretical Evaluation"):
        s_idx = all_samples[i]
        s_idx_2d = s_idx.reshape(1, -1) # Shape (1, n) for .score()

        # Calculate HT Estimate for this specific sample
        ht_est = np.sum(y_values[s_idx] / pik[s_idx]) #
        sum_prob += all_probs[i]
        # Calculate Spatial Indices using your Python modules
        results.append({
            'prob': all_probs[i],
            'ht': ht_est,
            'D': density_scorer.score(s_idx_2d)[0],
            'M': moran_scorer.score(s_idx_2d)[0],
            'V': voronoi_scorer.score(s_idx_2d)[0],
            'L': lb_scorer.score(s_idx_2d)[0]
        })
        # print('i', i , all_probs[i], sum_prob)
    df = pd.DataFrame(results)
    
    # 5. Calculate Theoretical Stats using Probability Weights
    def get_stats(column_name):
        # Expected Value: E[X] = sum(x * p)
        exp_val = np.sum(df[column_name] * df['prob'])
        # Variance: Var(X) = sum( (x - E[X])^2 * p )
        variance = np.sum(((df[column_name] - exp_val)**2) * df['prob'])
        return exp_val, np.sqrt(variance)

    # Calculate for all indices
    dm, ds = get_stats('D')
    mm, ms = get_stats('M')
    vm, vs = get_stats('V')
    lm, ls = get_stats('L')
    
    # Calculate Theoretical HT Variance
    # Note: E[HT] should equal true_total, so we use true_total for better precision
    ht_variance = np.sum(((df['ht'] - true_total)**2) * df['prob'])
    expec = np.sum(df['ht'] * df['prob'])
    # print('hereeee',ht_variance, len(pik)**2 * (1-np.sum(pik)/len(pik))) * np.var(y_values)/np.sum(pik)
    summary = {
        'Exp_Density': dm, 'SD_Density': ds,
        'Exp_Moran': mm,   'SD_Moran': ms,
        'Exp_Voronoi': vm, 'SD_Voronoi': vs,
        'Exp_Local': lm,   'SD_Local': ls,
        'HT_Variance': ht_variance
    }
    # --- DEBUG BLOCK ---
    S2_y = np.var(y_values) # Population Variance
    theoretical_srs_var = (N**2) * (1 - n/N) * (S2_y / n)
    print(f"DEBUG: Theoretical SRS Variance Bardia: {theoretical_srs_var:.2f}")
    print(f"DEBUG: Theoretical BRD Variance Bardia:{np.sum(y_values)} , {expec}, {np.sum(pik)}, {ht_variance:.2f}")
   
    return df, summary

### Optimized Sampling Functions

In [35]:
# B
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import localconverter

# Define the converter context for rpy2
combined_converter = ro.default_converter + numpy2ri.converter + pandas2ri.converter

def run_sampling_design(method, coords, probs, n, num_samples):
    N = len(coords)
    
    # 1. Python Methods (Nmcs and Rand)
    if method == "Nmcs":
        # Create the population and sampler
        pop = Population(coords=coords, probs=probs)
        sampler = KMeansSampler(
            population=pop, 
            n=n, 
            n_zones=(3, 3), 
            zone_builder="sweep",
            units_order="spiral",
            zones_order="spiral"
        )
        # Return samples for simulation AND the sampler for math
        return sampler.sample(num_samples), sampler

    if method == "Rand":
        samples_idx = np.zeros((num_samples, n), dtype=int)
        for i in range(num_samples):
            samples_idx[i] = np.random.choice(N, n, replace=False) #
        return samples_idx, None

    # 2. R Methods (Lopi, Wave, Maxe, Scps)
    samples_idx = np.zeros((num_samples, n), dtype=int)
    
    with localconverter(combined_converter):
        ro.globalenv['coords_r'] = coords
        ro.globalenv['probs_r'] = probs
        
        ro.r("library(BalancedSampling)")
        ro.r("library(WaveSampling)")
        ro.r("library(sampling)")
        
        for i in range(num_samples):
            if method == "Lopi":
                samples_idx[i] = np.array(ro.r("lpm2(probs_r, coords_r)")) - 1 #
            elif method == "Scps":
                samples_idx[i] = np.array(ro.r("scps(probs_r, coords_r)")) - 1 #
            elif method == "Wave":
                mask = ro.r("wave(coords_r, probs_r)") #
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0] #
            elif method == "Maxe":
                mask = ro.r("sampling::UPmaxentropy(probs_r)") #
                samples_idx[i] = np.where(np.array(mask).astype(bool))[0] #

    return samples_idx, None

### Metrics and Spread Calculation

In [36]:
# C
import numpy as np

def calculate_all_scores(coords, probs, sample_idx, n, N, scorers, y_val, method):
    """
    Calculates spatial and statistical scores using Python modules.
    
    Args:
        coords: Population coordinates.
        probs: Inclusion probabilities.
        sample_idx: 1D array of selected unit indices.
        n: Sample size.
        N: Population size.
        scorers: Dictionary of initialized Density, Moran, Voronoi, and LocalBalance objects.
        y_val: The variable of interest for estimation.
        method: The name of the sampling method.
    """
    # 1. Prepare index for Python (ensure 2D shape for .score method)
    s_idx_2d = sample_idx.reshape(1, -1)
    
    # 2. Estimator Calculation
    if method == "Rand":
        # SRS Estimator for random sampling
        est_val = N * np.mean(y_val[sample_idx])
    else:
        # HT Estimator: Sum(y_i / pi_i)
        est_val = np.sum(y_val[sample_idx] / probs[sample_idx])
    
    # 3. Calculate Spatial Scores using Python modules
    # Accessing pre-initialized scorers from the dictionary to avoid re-calculation
    dens_score = scorers['D'].score(s_idx_2d)[0]
    moran_score = scorers['M'].score(s_idx_2d)[0]
    voronoi_score = scorers['V'].score(s_idx_2d)[0]
    lb_score = scorers['L'].score(s_idx_2d)[0]

    # Return Order: Density, Voronoi, Moran, Local Balance, Estimator
    return (
        dens_score, 
        voronoi_score, 
        moran_score, 
        lb_score, 
        est_val
    )

### The Main Execution Loop

In [37]:

import pandas as pd
import os

folder = "/config/ws/graphical-sampling/populations"
populations = ['clust', 'grid', 'rand', 'meuse', 'swiss']

dfs = {}
for pop in populations:
    path = os.path.join(folder, f'{pop}.csv')
    dfs[pop] = pd.read_csv(path)
    # Changed 'df' to 'dfs[pop]' to match the loaded data
    corr_matrix = dfs[pop].corr(numeric_only=True)
    
    # Updated print to show the name and the matrix clearly
    print('\n', pop, 'Correlation Matrix = \n', corr_matrix.round(2))


 clust Correlation Matrix = 
          x     y  prob  z.70  z.80  z.90
x     1.00  0.22  0.99  0.69  0.79  0.90
y     0.22  1.00  0.29  0.18  0.39  0.28
prob  0.99  0.29  1.00  0.70  0.80  0.90
z.70  0.69  0.18  0.70  1.00  0.55  0.68
z.80  0.79  0.39  0.80  0.55  1.00  0.69
z.90  0.90  0.28  0.90  0.68  0.69  1.00

 grid Correlation Matrix = 
         x     y  prob  z.70  z.80  z.90
x     1.0 -0.00  1.00  0.70  0.80  0.90
y    -0.0  1.00  0.05 -0.07  0.08  0.05
prob  1.0  0.05  1.00  0.70  0.80  0.90
z.70  0.7 -0.07  0.70  1.00  0.55  0.68
z.80  0.8  0.08  0.80  0.55  1.00  0.69
z.90  0.9  0.05  0.90  0.68  0.69  1.00

 rand Correlation Matrix = 
         x     y  prob  z.70  z.80  z.90
x     1.0  0.00   1.0  0.70  0.80  0.90
y     0.0  1.00  -0.0  0.06 -0.04 -0.18
prob  1.0 -0.00   1.0  0.70  0.80  0.90
z.70  0.7  0.06   0.7  1.00  0.55  0.68
z.80  0.8 -0.04   0.8  0.55  1.00  0.69
z.90  0.9 -0.18   0.9  0.68  0.69  1.00

 meuse Correlation Matrix = 
             x     y  copper  ca

In [38]:
# D
# --- High-Efficiency Main Configuration and Loop ---
import os
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

# 1. Configuration Switches & Setup
INCLUDE_NMCS_IN_SIM = True  # Use theoretical math for Nmcs to save time
FIND_PIK_EXPLOSION = False
sample_cnt = 10000            # Increased sample count improves simulation stability
n_size = 5
pop_names = ["meuse"]        # Add "meuse", "swiss", etc. here

folder = "/config/ws/graphical-sampling/populations"
results_folder = "/config/ws/graphical-sampling/simulations/results"
os.makedirs(results_folder, exist_ok=True)

# Silence rpy2 warnings
warnings.filterwarnings("ignore", category=UserWarning, module="rpy2")

for name in pop_names:
    # 2. Load and Prep Data
    file_path = os.path.join(folder, f"{name}.csv")
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue
        
    df = pd.read_csv(file_path)
    coords = df[["x", "y"]].values.astype(float)
    
    # Assign target variable and inclusion probabilities
    if name == 'meuse':
        y_values = df["cadmium"].values
        pik = inclusion_probabilities(df["copper"].values, n_size)
    elif name == 'swiss':
        y_values = df["AREA_A"].values
        pik = inclusion_probabilities(df["AREA"].values, n_size)
    else:
        # Default for 'rand' or others
        y_values = df['z.90'].values
        pik = inclusion_probabilities(df["prob"].values, n_size)
    N = len(df)
    n = int(np.round(np.sum(pik)))

    # y_values = np.ones(N)  
    true_sum_y = np.sum(y_values)

    rho = np.corrcoef(y_values, pik)[0, 1]    
    print(f"\n--- Processing {name} (N={N}, n={n}, True Total={true_sum_y:.3f}) ---")

    # --- DEBUG BLOCK: Theoretical SRS Benchmark ---
    S2_y = np.var(y_values, ddof=1) 
    theoretical_srs_var = (N**2) * (1 - n/N) * (S2_y / n)
    print(f"DEBUG: Theoretical SRS Total Variance: {theoretical_srs_var:.2f}")

    # 3. Setup Metrics (One-time initialization)
    pop_wrapped = Population(coords=coords, probs=pik)
    scorer_D = Density(population=pop_wrapped, k=n)
    scorer_M = Moran(population=pop_wrapped)
    scorer_V = Voronoi(population=pop_wrapped)
    scorer_L = LocalBalance(population=pop_wrapped)

    # 4. Sampling and Scoring Loop
    all_data = []
    methods = ["Lopi", "Scps", "Rand"]# "Maxe", "Rand", 'Wave']
    if INCLUDE_NMCS_IN_SIM:
        methods.insert(0, "Nmcs")


    for m in methods:
        print(f"Running simulation for {m}...")
        # Get samples from Cell B
        samples, _ = run_sampling_design(m, coords, pik, n, sample_cnt)
        
        # Vectorized scoring (High Efficiency)
        d_scores = scorer_D.score(samples)
        m_scores = scorer_M.score(samples)
        v_scores = scorer_V.score(samples)
        l_scores = scorer_L.score(samples)
        
        # Calculate Estimators
        if m == "Rand":
            # Expansion Estimator for SRS
            est_scores = N * np.mean(y_values[samples], axis=1)
        else:
            # HT Estimator for PPS designs
            est_scores = np.sum(y_values[samples] / pik[samples], axis=1)

        # Store results
        for i in range(sample_cnt):
            all_data.append([m, d_scores[i], v_scores[i], m_scores[i], l_scores[i], est_scores[i]])
            # --- TRACKING RARE EXTREME ESTIMATES ---
            
            if FIND_PIK_EXPLOSION:
                error_margin = abs(est_scores[i] - true_sum_y) / true_sum_y

                if error_margin > 0.5:  # If estimate is >50% off the truth
                    print(f"\n[ALERT] Method: {m} | Iteration: {i}")
                    print(f"Estimate: {est_scores[i]:.2f} | True Total: {true_sum_y:.2f} | Error: {error_margin:.2%}")
                    
                    # Identify the specific units in this "bad" sample
                    bad_sample_idx = samples[i]
                    bad_y = y_values[bad_sample_idx]
                    bad_pik = pik[bad_sample_idx]
                    
                    # Find which specific unit caused the explosion
                    # The one with the highest y/pik ratio is usually the culprit
                    contributions = bad_y / bad_pik
                    culprit_local_idx = np.argmax(contributions)
                    culprit_pop_idx = bad_sample_idx[culprit_local_idx]
                    
                    print(f"Sample Indices: {bad_sample_idx}")
                    print(f"Exploded Unit Index: {culprit_pop_idx} | y: {bad_y[culprit_local_idx]} | pik: {bad_pik[culprit_local_idx]}")
# ---------------------------------------
    # 5. Handle Nmcs Theoretical Math (Cell A Injection)
    theoretical_results = None
    if not INCLUDE_NMCS_IN_SIM:
        print("Calculating Theoretical Math for Nmcs...")
        _, nmcs_sampler = run_sampling_design("Nmcs", coords, pik, n, sample_cnt)
        # We pass y_values explicitly to Cell A
        _, theoretical_results = evaluate_sampler(nmcs_sampler, y_values=y_values)

    # 6. Results Processing
    res_df = pd.DataFrame(all_data, columns=["Method", "D", "V", "M", "L", "HT"])
    
    # Aggregate simulation results
    summary = res_df.groupby("Method").agg({
        "D": ["mean", "std"], 
        "V": ["mean", "std"], 
        "M": ["mean", "std"], 
        "L": ["mean", "std"], 
        "HT": ["mean", "var"]
    })

    # CRITICAL: Force column order before renaming to ensure Math matches Sim columns
    summary = summary[["D", "V", "M", "L", "HT"]]
    summary.columns = ["Dm", "Ds", "Vm", "Vs", "Mm", "Ms", "Lm", "Ls", "HTm", "HTv"]

    # Inject Theoretical Nmcs row
    if theoretical_results:
        t = theoretical_results
        summary.loc["Nmcs"] = [
            t['Exp_Density'], t['SD_Density'], 
            t['Exp_Voronoi'], t['SD_Voronoi'], 
            t['Exp_Moran'], t['SD_Moran'], 
            t['Exp_Local'], t['SD_Local'], 
            true_sum_y, t['HT_Variance']
        ]

    # 7. Final Metrics: RB and Efficiency (Eff)
    summary["RB"] = (summary["HTm"] - true_sum_y) / true_sum_y
    if "Rand" in summary.index:
        rand_var = summary.loc["Rand", "HTv"]
        summary["Eff"] = rand_var / summary["HTv"].replace(0, np.nan)
        summary['rho'] = rho
    # Final Table Output
    final_cols = ['rho', "HTm", "HTv" ,"Eff", "RB", "Dm", "Ds", "Vm", "Vs", "Mm", "Ms", "Lm", "Ls"]
    # Reorder to ensure RB and Eff are visible next to Estimator mean
    summary = summary.reindex(columns=summary.columns.tolist())# + ["RB", "Eff"])
    print(summary[final_cols].round(3))

    # Save results
    summary.to_csv(os.path.join(results_folder, f"summary_{name}.csv"))


--- Processing meuse (N=155, n=5, True Total=503.100) ---
DEBUG: Theoretical SRS Total Variance: 57738.05
Running simulation for Nmcs...


/config/ws/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server"
  warnings.warn(
/config/ws/graphical-sampling/.venv/lib/python3.12/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "R_LIBS_SITE" redefined by R and overriding existing variable. Current: "/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library/:/usr/local/lib/R/site-library:/usr/lib/R/site-library:/usr/lib/R/library:/usr

Running simulation for Lopi...
Running simulation for Scps...
Running simulation for Rand...
          rho      HTm        HTv    Eff     RB     Dm     Ds     Vm     Vs  \
Method                                                                        
Lopi    0.925  501.664  11301.622  5.020 -0.003  0.088  0.189  0.125  0.078   
Nmcs    0.925  501.774  15251.911  3.720 -0.003  0.047  0.064  0.072  0.054   
Rand    0.925  502.708  56738.892  1.000 -0.001 -0.081  0.333  0.364  0.298   
Scps    0.925  505.305  11007.195  5.155  0.004  0.090  0.180  0.112  0.072   

           Mm     Ms     Lm     Ls  
Method                              
Lopi   -0.180  0.076  0.519  0.177  
Nmcs   -0.216  0.092  0.537  0.228  
Rand   -0.052  0.096  0.711  0.217  
Scps   -0.202  0.067  0.507  0.167  


# Store

In [39]:
# This cell acts as a barrier
raise SystemExit("Stopping Run All: Archived cells below.")

SystemExit: Stopping Run All: Archived cells below.

/config/ws/graphical-sampling/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:

import pandas as pd
import os

folder = "/config/ws/graphical-sampling/populations"
populations = ['clust', 'grid', 'rand', 'meuse', 'swiss']

dfs = {}
for pop in populations:
    path = os.path.join(folder, f'{pop}.csv')
    dfs[pop] = pd.read_csv(path)
    # Changed 'df' to 'dfs[pop]' to match the loaded data
    corr_matrix = dfs[pop].corr(numeric_only=True)
    
    # Updated print to show the name and the matrix clearly
    print('\n', pop, 'Correlation Matrix = \n', corr_matrix.round(2))


 clust Correlation Matrix = 
          x     y  prob  z.70  z.80  z.90
x     1.00  0.22  0.99  0.69  0.79  0.90
y     0.22  1.00  0.29  0.18  0.39  0.28
prob  0.99  0.29  1.00  0.70  0.80  0.90
z.70  0.69  0.18  0.70  1.00  0.55  0.68
z.80  0.79  0.39  0.80  0.55  1.00  0.69
z.90  0.90  0.28  0.90  0.68  0.69  1.00

 grid Correlation Matrix = 
         x     y  prob  z.70  z.80  z.90
x     1.0 -0.00  1.00  0.70  0.80  0.90
y    -0.0  1.00  0.05 -0.07  0.08  0.05
prob  1.0  0.05  1.00  0.70  0.80  0.90
z.70  0.7 -0.07  0.70  1.00  0.55  0.68
z.80  0.8  0.08  0.80  0.55  1.00  0.69
z.90  0.9  0.05  0.90  0.68  0.69  1.00

 rand Correlation Matrix = 
         x     y  prob  z.70  z.80  z.90
x     1.0  0.00   1.0  0.70  0.80  0.90
y     0.0  1.00  -0.0  0.06 -0.04 -0.18
prob  1.0 -0.00   1.0  0.70  0.80  0.90
z.70  0.7  0.06   0.7  1.00  0.55  0.68
z.80  0.8 -0.04   0.8  0.55  1.00  0.69
z.90  0.9 -0.18   0.9  0.68  0.69  1.00

 meuse Correlation Matrix = 
             x     y  copper  ca

In [ ]:
import pandas as pd
import os

folder = "/config/ws/graphical-sampling/populationssss"

# --- 1. Process Meuse ---
meuse_path = os.path.join(folder, "meuse_full.csv")
if os.path.exists(meuse_path):
    df_meuse = pd.read_csv(meuse_path)
    # Keep only the requested columns
    df_meuse = df_meuse[['x', 'y', 'copper', 'cadmium', 'lead', 'zinc']]
    # Save back to folder
    df_meuse.to_csv(os.path.join(folder, "meuse.csv"), index=False)
    print("✅ Meuse modified and saved.")

# --- 2. Process Swiss ---
swiss_path = os.path.join(folder, "swiss_full.csv")
if os.path.exists(swiss_path):
    df_swiss = pd.read_csv(swiss_path)
    # Keep specific columns
    df_swiss = df_swiss[['COORD_X', 'COORD_Y', 'AREA', 'AREA_A', 'AREA_B']]
    # Rename coordinates to x and y
    df_swiss = df_swiss.rename(columns={'COORD_X': 'x', 'COORD_Y': 'y'})
    # Save back to folder
    df_swiss.to_csv(os.path.join(folder, "swiss.csv"), index=False)
    print("✅ Swiss modified and saved.")

In [ ]:
import numpy as np
import pandas as pd
import os

# 1. Define your file list based on the image provided
# Note: I am assuming the folder name is 'data_samples' based on previous context.
# If they are in the current directory, change folder to "."
folder = "/config/ws/graphical-sampling/populations"

pop_files = {
    'clust_eq':   'clust_eq_N=100.csv',
    'clust_uneq': 'clust_uneq_N=100.csv',
    'grid_eq':    'grid_eq_N=100.csv',
    'grid_uneq':  'grid_uneq_N=100.csv',
    'random_eq':  'random_eq_N=100.csv',
    'random_uneq':'random_uneq_N=100.csv',
}

# Helper function to generate correlated variables
def generate_correlated_variable(v, correlation, seed=None):
    """Generates a new variable correlated with vector v at a specific r."""
    if seed: np.random.seed(seed)
    # 1. Create random noise
    noise = np.random.normal(0, 1, len(v))
    
    # 2. Standardize target v to remove mean/scale effects for calculation
    v_norm = (v - np.mean(v)) / np.std(v)
    
    # 3. Residualize noise (make it orthogonal to v)
    # This step ensures exact mathematical control over correlation
    noise_resid = noise - (np.dot(noise, v_norm) / np.dot(v_norm, v_norm)) * v_norm
    noise_norm = noise_resid / np.std(noise_resid)
    
    # 4. Combine to get desired correlation
    # New = r * Old + sqrt(1-r^2) * Noise
    new_var = correlation * v_norm + np.sqrt(1 - correlation**2) * noise_norm
    
    # 5. Rescale back to original range (optional, but good for probability-like vars)
    # Here we just shift it to be positive to act as a "size" variable
    new_var = new_var - np.min(new_var) + 0.1 
    return new_var

data_store = {}

print("--- Loading & Processing Data ---")
for key, fname in pop_files.items():
    path = os.path.join(folder, fname)
    
    if os.path.exists(path):
        df = pd.read_csv(path)
        
        # Basic extractions
        coords = df[['x', 'y']].values
        probs = df['prob'].values
        N = len(df)
        
        # --- LOGIC: Handle "uneq" vs "eq" files ---
        # We look for "uneq" in the key name
        if "uneq" in key:
            print(f"Processing {key}: Generating correlated auxiliaries...")
            
            # Generate the 3 auxiliary variables
            # These act as 'Target Y' variables with different correlations to inclusion probs
            y_70 = generate_correlated_variable(probs, 0.70, seed=42)
            y_80 = generate_correlated_variable(probs, 0.80, seed=43)
            y_90 = generate_correlated_variable(probs, 0.90, seed=44)
            
            # Store them so we can loop over them later
            targets_dict = {
                'y_70': y_70,
                'y_80': y_80,
                'y_90': y_90
            }
        else:
            # For Equal Probability (EP), Prob is constant. 
            # Correlation with a constant is undefined/zero. 
            # We just create one synthetic target to test spatial balance.
            print(f"Processing {key}: Standard EP file.")
            synthetic_y = (df['x'] + df['y']) + np.random.normal(0, 1, N)
            targets_dict = {'y_synthetic': synthetic_y}

        # Save to data_store
        data_store[key] = {
            'coords': coords,
            'probs': probs,
            'targets': targets_dict, # Now holds multiple Ys
            'N': N
        }
    else:
        print(f"⚠️ Warning: File {fname} not found in {folder}. Skipping.")
        
print("✅ Data Loaded Successfully.")

--- Loading & Processing Data ---
Processing clust_eq: Standard EP file.
Processing clust_uneq: Generating correlated auxiliaries...
Processing grid_eq: Standard EP file.
Processing grid_uneq: Generating correlated auxiliaries...
Processing random_eq: Standard EP file.
Processing random_uneq: Generating correlated auxiliaries...
✅ Data Loaded Successfully.


In [ ]:
# Save the modified data to new CSVs
output_folder = "modified_populations"
os.makedirs(output_folder, exist_ok=True)

print("\n--- Saving All Generated Variables ---")
for key, data in data_store.items():
    # 1. Start with the base spatial and probability data
    df_output = pd.DataFrame(data['coords'], columns=['x', 'y'])
    df_output['prob'] = data['probs']
    
    # 2. Add every target stored in the dictionary
    for target_name, target_values in data['targets'].items():
        # Map the internal name (y_70) to your requested format (z.70)
        # We split by underscore and join with a dot
        formatted_name = target_name.replace("y_", "z.") 
        
        df_output[formatted_name] = target_values
        
    # 3. Save the file
    save_path = os.path.join(output_folder, f"{key}_modified.csv")
    df_output.to_csv(save_path, index=False)
    
    # 4. Feedback on what was saved
    saved_cols = [c for c in df_output.columns if 'z.' in c or 'synthetic' in c]
    print(f"Saved {key}: Included variables {saved_cols}")

print(f"\n✅ All files saved successfully in: {output_folder}")


--- Saving All Generated Variables ---
Saved clust_eq: Included variables ['z.synthetic']
Saved clust_uneq: Included variables ['z.70', 'z.80', 'z.90']
Saved grid_eq: Included variables ['z.synthetic']
Saved grid_uneq: Included variables ['z.70', 'z.80', 'z.90']
Saved random_eq: Included variables ['z.synthetic']
Saved random_uneq: Included variables ['z.70', 'z.80', 'z.90']

✅ All files saved successfully in: modified_populations


In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(sp)
data(meuse)        # Loads the dataframe
data(meuse.grid)   # Loads the prediction grid
data(meuse.riv)    # Loads the river boundaries

# Convert to simple features (modern format)
library(sf)
meuse_sf <- st_as_sf(meuse, coords = c("x", "y"), crs = 28992)
meuse_sf

Simple feature collection with 155 features and 12 fields
Geometry type: POINT
Dimension:     XY
Bounding box:  xmin: 178605 ymin: 329714 xmax: 181390 ymax: 333611
Projected CRS: Amersfoort / RD New
First 10 features:
   cadmium copper lead zinc  elev       dist   om ffreq soil lime landuse
1     11.7     85  299 1022 7.909 0.00135803 13.6     1    1    1      Ah
2      8.6     81  277 1141 6.983 0.01222430 14.0     1    1    1      Ah
3      6.5     68  199  640 7.800 0.10302900 13.0     1    1    1      Ah
4      2.6     81  116  257 7.655 0.19009400  8.0     1    2    0      Ga
5      2.8     48  117  269 7.480 0.27709000  8.7     1    2    0      Ah
6      3.0     61  137  281 7.791 0.36406700  7.8     1    2    0      Ga
7      3.2     31  132  346 8.217 0.19009400  9.2     1    2    0      Ah
8      2.8     29  150  406 8.490 0.09215160  9.5     1    1    0      Ab
9      2.4     37  133  347 8.668 0.18461400 10.6     1    1    0      Ab
10     1.6     24   80  183 9.049 0.309702

Linking to GEOS 3.12.1, GDAL 3.8.4, PROJ 9.4.0; sf_use_s2() is TRUE


In [ ]:
%%R
# Drop geometry
meuse_df <- st_drop_geometry(meuse_sf)

# Keep only numeric columns
numeric_vars <- meuse_df[sapply(meuse_df, is.numeric)]

# Compute correlation matrix
cor_matrix <- cor(numeric_vars, use = "complete.obs")

cor_matrix


           cadmium     copper       lead       zinc       elev       dist
cadmium  1.0000000  0.9255639  0.7984435  0.9163334 -0.5651759 -0.6167057
copper   0.9255639  1.0000000  0.8166826  0.9074859 -0.5816771 -0.6112932
lead     0.7984435  0.8166826  1.0000000  0.9543124 -0.5882216 -0.5804984
zinc     0.9163334  0.9074859  0.9543124  1.0000000 -0.5970112 -0.6469027
elev    -0.5651759 -0.5816771 -0.5882216 -0.5970112  1.0000000  0.5310513
dist    -0.6167057 -0.6112932 -0.5804984 -0.6469027  0.5310513  1.0000000
om       0.7307845  0.7347169  0.5535007  0.6842578 -0.3561615 -0.5668039
dist.m  -0.6207236 -0.6160603 -0.5881192 -0.6599304  0.5092804  0.9840138
                om     dist.m
cadmium  0.7307845 -0.6207236
copper   0.7347169 -0.6160603
lead     0.5535007 -0.5881192
zinc     0.6842578 -0.6599304
elev    -0.3561615  0.5092804
dist    -0.5668039  0.9840138
om       1.0000000 -0.5890220
dist.m  -0.5890220  1.0000000


In [ ]:
%%R
# Extract coordinates
coords <- st_coordinates(meuse_sf)

# Drop geometry and bind coordinates
meuse_df <- cbind(
  st_drop_geometry(meuse_sf),
  x = coords[,1],
  y = coords[,2]
)

# Convert to plain data frame (optional but safe)
meuse_df <- as.data.frame(meuse_df)

head(meuse_df)



  cadmium copper lead zinc  elev       dist   om ffreq soil lime landuse dist.m
1    11.7     85  299 1022 7.909 0.00135803 13.6     1    1    1      Ah     50
2     8.6     81  277 1141 6.983 0.01222430 14.0     1    1    1      Ah     30
3     6.5     68  199  640 7.800 0.10302900 13.0     1    1    1      Ah    150
4     2.6     81  116  257 7.655 0.19009400  8.0     1    2    0      Ga    270
5     2.8     48  117  269 7.480 0.27709000  8.7     1    2    0      Ah    380
6     3.0     61  137  281 7.791 0.36406700  7.8     1    2    0      Ga    470
       x      y
1 181072 333611
2 181025 333558
3 181165 333537
4 181298 333484
5 181307 333330
6 181390 333260


In [ ]:
%%R
# Extract coordinates and drop geometry column
coords <- st_coordinates(meuse_sf)
meuse_df <- cbind(st_drop_geometry(meuse_sf), x = coords[,1], y = coords[,2])

# Save it as a CSV (in your working directory)
write.csv(meuse_df, "meuse_with_coords.csv", row.names = FALSE)


In [ ]:
import os
import pandas as pd

# Load your final CSV from R
meuse_df = pd.read_csv("meuse_with_coords.csv")

# Create folder
output_folder = "modified_populations"
os.makedirs(output_folder, exist_ok=True)

# Save it
meuse_df.to_csv(f"{output_folder}/meuse_modified.csv", index=False)

print("Saved successfully.")


Saved successfully.
